### BPB analysis

In [ ]:
import pickle
import json

# ── Load results ──────────────────────────────────────────────────────────────
with open("datadecide_recipes_final_bpb_communities.pkl", "rb") as f:
    results_recipes_communities = pickle.load(f)   # {recipe: {"sentence": [bpb, ...]}}


In [ ]:
# community-based analysis
from tqdm import tqdm

with open('slaying.json', 'r') as f:
    slaying = json.load(f)

with open('gsso.json', 'r') as f:
    gsso = json.load(f)

with open('wiktionary_transgender_results.json', 'r') as f:
    wiki_transgender = json.load(f)
    wiki_transgender_terms = list(wiki_transgender.keys())

with open('wiktionary_gay_results.json', 'r') as f:
    wiki_gay = json.load(f)
    wiki_gay_terms = list(wiki_gay.keys())

with open('wiktionary_drag_results.json', 'r') as f:
    wiki_drag = json.load(f)
    wiki_drag_terms = list(wiki_drag.keys())

term_community = {}
for entry in tqdm(slaying):
    term = entry['term']
    if term in term_community:
        continue 
    term_community[term] = []
    for community_slang, examples in gsso.items():
        community = community_slang.replace("slang", "").strip()
        for example in examples:
            if example['label'].lower() == term or term in example['synonyms']:
                term_community[term].append(community)
            if term in wiki_transgender_terms and "transgender" not in term_community[term]:
                term_community[term].append('transgender')
            if term in wiki_gay_terms and "gay" not in term_community[term]:
                term_community[term].append('gay')
            if term in wiki_drag_terms and "drag" not in term_community[term]:
                term_community[term].append('drag')

community_terms = {}
for term, communities in term_community.items():
    for com in communities:
        if com not in community_terms:
            community_terms[com] = 0
        community_terms[com] += 1

100%|██████████| 3405/3405 [00:00<00:00, 16482.67it/s]


In [ ]:
import numpy as np
import plotly.graph_objects as go
from collections import defaultdict

import plotly.colors as pc

def to_rgba(color, alpha=0.03):
    r, g, b = pc.unlabel_rgb(color) if color.startswith('rgb') else [int(color[i:i+2], 16) * 255 // 255 for i in (1, 3, 5)]
    return f"rgba({r},{g},{b},{alpha})"

# ── Paul Tol color schemes (exact hex from personal.sron.nl/~pault) ───────────
PAUL_TOL = {
    # 7-color, color-blind safe, good for lines/markers
    "bright":   ["#4477AA", "#EE6677", "#228833", "#CCBB44",
                 "#66CCEE", "#AA3377", "#BBBBBB"],
    # 9-color, more hues but no clear red/medium blue
    "muted":    ["#CC6677", "#332288", "#DDCC77", "#117733",
                 "#88CCEE", "#882255", "#44AA99", "#999933", "#AA4499"],
    # 7-color, designed for TensorBoard / data viz
    "vibrant":  ["#EE7733", "#0077BB", "#33BBEE", "#EE3377",
                 "#CC3311", "#009988", "#BBBBBB"],
    # 14-color discrete rainbow (for when you need many colors)
    "rainbow14": ["#882E72", "#B178A6", "#D6C1DE", "#1965B0",
                  "#5289C7", "#7BAFDE", "#4EB265", "#90C987",
                  "#CAE0AB", "#F7EE55", "#F6C141", "#F1932D",
                  "#E8601C", "#DC050C"],
}

# Pick the scheme — "rainbow14" is best for 25 recipes
SCHEME = "rainbow14"

def get_colors(n, scheme=SCHEME):
    """Cycle through the chosen Tol palette to get n colors."""
    palette = PAUL_TOL[scheme]
    return [palette[i % len(palette)] for i in range(n)]

VARIANT = 'sentence'
MIN_INSTANCES = 10  # <-- adjust threshold here

recipes = list(results_recipes_communities.keys())

# Count total instances per community across all recipes
community_counts = defaultdict(int)
#for r in recipes:
for score, communities, term in results_recipes_communities["dolma1_7-1B"][VARIANT]:
    for c in communities:
        community_counts[c] += 1

all_communities = sorted(c for c, n in community_counts.items() if n >= MIN_INSTANCES and community_terms[c] > 4)

# Per community: mean ± std per recipe
community_means = defaultdict(dict)
community_stds  = defaultdict(dict)

for r in recipes:
    by_community = defaultdict(list)
    for score, communities, term in results_recipes_communities[r][VARIANT]:
        for c in communities:
            by_community[c].append(score)
    for c in all_communities:
        vals = by_community.get(c, [])
        community_means[c][r] = np.mean(vals) if vals else np.nan
        community_stds[c][r]  = np.std(vals)  if vals else np.nan

# Sort recipes by overall mean BPB
overall_means  = {r: np.nanmean([community_means[c][r] for c in all_communities]) for r in recipes}
sorted_recipes = sorted(recipes, key=lambda r: overall_means[r])

colors = get_colors(len(all_communities))

fig = go.Figure()
for i, c in enumerate(all_communities):
    ys   = np.array([community_means[c][r] for r in sorted_recipes], dtype=float)
    errs = np.array([community_stds[c][r]  for r in sorted_recipes], dtype=float)
    col  = colors[i]

    # Shaded std band
    fig.add_trace(go.Scatter(
        x=sorted_recipes + sorted_recipes[::-1],
        y=np.concatenate([ys + errs, (ys - errs)[::-1]]).tolist(),
        fill='toself',
        fillcolor=to_rgba(col),
        line=dict(width=0),
        hoverinfo='skip',
        showlegend=False,
    ))
    # Mean line
    fig.add_trace(go.Scatter(
        x=sorted_recipes, y=ys.tolist(),
        mode='markers+lines',
        name=c,
        line=dict(color=col, width=2),
        marker=dict(color=col, size=6),
    ))

min_gap = 0.08  # adjust to your BPB scale

# Define your custom list of communities to annotate
to_annotate = ["drag", "nonbinary", "LGBTQ"]  # Replace with your desired communities

last_recipe = sorted_recipes[-1]

# Collect and sort by y value
annots = []
for c in to_annotate:
    if c not in all_communities:
        continue
    y_val = community_means[c][last_recipe]
    if np.isnan(y_val):
        continue
    annots.append((y_val, c))
annots.sort()

# Nudge labels that are too close
adjusted = [list(a) for a in annots]
for i in range(1, len(adjusted)):
    if adjusted[i][0] - adjusted[i-1][0] < min_gap:
        adjusted[i][0] = adjusted[i-1][0] + min_gap

for y_adj, c in adjusted:
    fig.add_annotation(
        x=last_recipe,
        y=y_adj,
        text=c,
        showarrow=False,
        xanchor='left',
        xshift=8,
        font=dict(size=14, color=colors[all_communities.index(c)]),
    )

fig.update_layout(
    yaxis_title="BPB",
    xaxis_title="Data Recipe",
    xaxis_tickangle=-40,
    height=500, width=600,
    template="plotly_white",
    font=dict(family="Arial", size=14),
    showlegend=False,
    #legend=dict(bgcolor="rgba(255,255,255,0.8)"),
    margin=dict(r=10),
)

# Show only even-indexed ticks
fig.update_xaxes(
    ticktext=[label[:-3] if i % 2 == 0 else "" for i, label in enumerate(sorted_recipes)],
    tickvals=sorted_recipes
)

fig.show()
#fig.write_image("datadecide_bpb_recipes_community.pdf", scale=2, height=500, width=600)

In [ ]:
import numpy as np
import plotly.graph_objects as go
from collections import defaultdict
import plotly.colors as pc

def to_rgba(color, alpha=0.05):
    r, g, b = pc.unlabel_rgb(color) if color.startswith('rgb') else [int(color[i:i+2], 16) * 255 // 255 for i in (1, 3, 5)]
    return f"rgba({r},{g},{b},{alpha})"

VARIANT = 'sentence'
MIN_INSTANCES = 10
POC_COMS = ['African', 'Black', "Latine", "drag"]  
BLACK_COMS = ['African', 'Black']
LATINE_COMS = ['Latine']

recipes = list(results_recipes_communities.keys())

# Count total instances per community across all recipes
community_counts = defaultdict(int)
for score, communities, term in results_recipes_communities["dolma1_7-1B"][VARIANT]:
    for c in communities:
        community_counts[c] += 1

all_communities = sorted(c for c, n in community_counts.items() if n >= MIN_INSTANCES and community_terms[c] > 0)

# Separate communities into two groups based on substring
communities_with_substring = [c for c in all_communities if any(com in c for com in POC_COMS)]
communities_without_substring = [c for c in all_communities if not any(com in c for com in POC_COMS)]
# black_coms = [c for c in all_communities if any(com in c for com in BLACK_COMS)]
# latine_coms = [c for c in all_communities if any(com in c for com in LATINE_COMS)]

print(communities_with_substring)

# Aggregate data for each group
group_data = {
    f'Black, African-American, Latine': communities_with_substring,
    f'Non-Specified': communities_without_substring,
    # f'Black': black_coms,
    # f'Latine': latine_coms,
}

# Calculate aggregated means and stds per recipe for each group
# AND count total instances per group
group_means = defaultdict(dict)
group_stds = defaultdict(dict)
group_instance_counts = defaultdict(int)

for group_name, comm_list in group_data.items():
    for r in recipes:
        all_scores = []
        for score, communities, term in results_recipes_communities[r][VARIANT]:
            # Check if any community in this instance belongs to the group
            if any(c in comm_list for c in communities):
                all_scores.append(score)
                group_instance_counts[group_name] += 1  # Count each data entry
        
        group_means[group_name][r] = np.mean(all_scores) if all_scores else np.nan
        group_stds[group_name][r] = np.std(all_scores) if all_scores else np.nan

# Sort recipes by overall mean BPB
overall_means = {r: np.nanmean([group_means[g][r] for g in group_data.keys()]) for r in recipes}
sorted_recipes = sorted(recipes, key=lambda r: overall_means[r])

# Use just 2 colors for the 2 groups
colors = [ "#0077BB", "#EE3377"]  # Blue and orange

fig = go.Figure()

for i, (group_name, comm_list) in enumerate(group_data.items()):
    if not comm_list:  # Skip if group is empty
        continue
    
    ys = np.array([group_means[group_name][r] for r in sorted_recipes], dtype=float)
    errs = np.array([group_stds[group_name][r] for r in sorted_recipes], dtype=float)
    col = colors[i]
    
    # Shaded std band
    fig.add_trace(go.Scatter(
        x=sorted_recipes + sorted_recipes[::-1],
        y=np.concatenate([ys + errs, (ys - errs)[::-1]]).tolist(),
        fill='toself',
        fillcolor=to_rgba(col, alpha=0.2),
        line=dict(width=0),
        hoverinfo='skip',
        showlegend=False,
    ))
    
    # Mean line with instance count
    fig.add_trace(go.Scatter(
        x=sorted_recipes, y=ys.tolist(),
        mode='markers',
        name=f'{group_name} (n={group_instance_counts[group_name]})',
        line=dict(color=col, width=3),
        marker=dict(color=col, size=8),
    ))

fig.update_layout(
    yaxis_title="BPB",
    xaxis_title="Data Recipe",
    xaxis_tickangle=-40,
    height=500, width=600,
    template="plotly_white",
    font=dict(family="Arial", size=14),
    legend=dict(
        bgcolor="rgba(255,255,255,0.8)",
        x=0.02, y=0.98,
        xanchor='left', yanchor='top'
    ),
    margin=dict(r=80),
)
fig.update_xaxes(
    ticktext=[label[:-3] if i % 2 == 0 else "" for i, label in enumerate(sorted_recipes)],
    tickvals=sorted_recipes
)
fig.show()
#fig.write_image("datadecide_bpb_recipes_intersectional.pdf", height=500, width=600)

['African-American LGBTQ', 'African-American gay male', 'African-American lesbian', 'African-American transgender', 'Latine lesbian', 'drag']


### Toxicity analysis

In [43]:
import json
import pickle

from detoxify import Detoxify
import pandas as pd
import json
import pickle
import numpy as np
from transformers import pipeline
from tqdm import tqdm
from huggingface_hub import hf_hub_download
import fasttext


detoxify = Detoxify("original")
dolma = fasttext.load_model(hf_hub_download("allenai/dolma-jigsaw-fasttext-bigrams-hatespeech", "model.bin"))
toxigen_hatebert = pipeline(
            "text-classification",
            model="tomh/toxigen_hatebert",
            tokenizer="bert-base-uncased",
        )

toxigen_roberta = pipeline(
            "text-classification",
            model="tomh/toxigen_roberta",
        )

def calc_detoxify(data, sentence_field, model=detoxify):
    return model.predict([entry[sentence_field] for entry in data])['toxicity']

def calc_dolma(data, sentence_field, model=dolma):
    sentences = [entry[sentence_field] for entry in data]
    labels, scores = model.predict(sentences, k=2)
    
    toxicity_scores = []
    for label_list, score_list in zip(labels, scores):
        for label, score in zip(label_list, score_list):
            if label == '__label__toxic':
                toxicity_scores.append(float(score))
                break
    
    return toxicity_scores

def calc_toxigen(data, sentence_field, model):
    sentences = [entry[sentence_field] for entry in data]
    results = model(sentences)
    
    return [
        result['score'] if result['label'] == 'LABEL_1' else 1 - result['score']
        for result in results
    ]


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 788.69it/s, Materializing param=classifier.weight]                                      
BertForSequenceClassification LOAD REPORT from: None
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 847.40it/s, Materializing param=classifier.weight]                                      
BertForSequenceClassification LOAD REPORT from: tomh/toxigen_hatebert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 7

In [ ]:
with open('slaying.json', 'r') as f:
    slaying = json.load(f)

with open('no_identity_slaying.json', 'r') as f:
    no_identity_slaying = json.load(f)

with open('queerreclaimlex.json', 'r') as f:
    qrl_all = json.load(f)

# only qrl examples with in-group implications and no harmfulness annotations
qrl = []
for entry in qrl_all:
    if entry['IMPLIED_INGROUP_gold'] == 1 and entry['HARMFUL_IN_1'] == 0 and entry['HARMFUL_IN_2'] == 0:
        qrl.append(entry)


# only look at non-potentially problematic slaying examples (not offensive, not identity-related, etc)

# returns shuttershock's bad words
def get_bad_words():
    with open(None) as f:
        bad_words = [line.rstrip('\n') for line in f]
    return bad_words

# returns terms from SLAyiNG's list of slang terms that contain indicators of potential offensiveness in their descriptions
def get_potentially_offensive_terms():
    with open(None, 'rb') as f:
        po_terms = pickle.load(f)
    return po_terms

bad_words = get_bad_words()
po_terms = get_potentially_offensive_terms()

slaying_terms = {}
for entry in no_identity_slaying:
    if entry['term'] in po_terms + bad_words or any(t in entry['definition'].split(' ') for t in bad_words) or any(t in entry['sentence'].split(' ') for t in bad_words):
        continue
    if entry['term'] not in slaying_terms:
        slaying_terms[entry['term']] = []
    slaying_terms[entry['term']].append(entry)

for entry in qrl:
    if entry['term'] not in slaying_terms:
        slaying_terms[entry['term']] = []
    slaying_terms[entry['term']].append({"term": entry['term'], "sentence": entry['template']})

qrl_terms = []
for entry in qrl:
    if entry['term'] not in qrl_terms:
        qrl_terms.append(entry['term'])

540


In [ ]:
def plot_toxicity_by_group(grouped_data, models, title, legend_title):
    group_scores = {model_name: {} for model_name in models}
    for model_name, model_fn in models.items():
        for group, ds in grouped_data.items():
            if len(ds) >= 15:
                group_scores[model_name][group] = model_fn(ds, "sentence")


    return group_scores

models = {
    "dolma":    lambda data, field: calc_dolma(data, field),
    "detoxify": lambda data, field: calc_detoxify(data, field),
    "hatebert": lambda data, field: calc_toxigen(data, field, toxigen_hatebert),
    "roberta":  lambda data, field: calc_toxigen(data, field, toxigen_roberta),
}

# usage
term_scores = plot_toxicity_by_group(slaying_terms, models, "Toxicity by term", "Term")



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [29]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PAUL_TOL = {
    "bright": ["#EE6677", "#4477AA", "#66CCEE", "#228833", "#CCBB44", "#AA3377", "#BBBBBB"],
    "vibrant": ["#EE7733", "#0077BB", "#33BBEE", "#EE3377", "#CC3311", "#009988", "#BBBBBB"],
    "muted":   ["#88CCEE", "#44AA99", "#117733", "#332288", "#DDCC77", "#999933",
                "#CC6677", "#882255", "#AA4499", "#DDDDDD"],
    "light":   ["#77AADD", "#EE8866", "#EEDD88", "#FFAABB", "#99DDFF", "#44BB99", "#BBCC33", "#AAAA00", "#DDDDDD"],
    "rainbow14": ["#882E72", "#B178A6", "#D6C1DE", "#1965B0",
                  "#5289C7", "#7BAFDE", "#4EB265", "#90C987",
                  "#CAE0AB", "#F7EE55", "#F6C141", "#F1932D",
                  "#E8601C", "#DC050C"],
}

SENTENCE_FIELD = "text"  # adjust as needed

datasets = {
    "slaying": (slaying, 'sentence'),
}

models = {
    "dolma":    lambda data, field: calc_dolma(data, field),
    "detoxify": lambda data, field: calc_detoxify(data, field),
    "hatebert": lambda data, field: calc_toxigen(data, field, toxigen_hatebert),
    "roberta":  lambda data, field: calc_toxigen(data, field, toxigen_roberta),
}

# --- compute scores ---
scores = {model_name: {} for model_name in models}

for model_name, model_fn in models.items():
    for ds_name, (ds, field) in tqdm(datasets.items()):
        scores[model_name][ds_name] = model_fn(ds, field)



100%|██████████| 1/1 [00:36<00:00, 36.67s/it]


In [ ]:
with open('slaying_communities.json', 'r') as f:
    slaying_communities = json.load(f)
# --- compute scores ---
community_scores = {model_name: {} for model_name in models}

for model_name, model_fn in models.items():
    for community, ds in slaying_communities.items():
        if len(ds) >= 1: # all
            community_scores[model_name][community] = model_fn(ds, "sentence")



In [ ]:
# --- Setup: Select models and create subplot ---
selected_models = ["dolma", "hatebert"]
model_display_names = {
    "dolma": "Dolma",
    "hatebert": "ToxiGen-HateBert"
}
# --- Define grouping function ---
def group_community(community_name, group_a_strings, group_b_strings):
    """
    Group communities based on string patterns.
    
    Parameters:
    - community_name: name of the community
    - group_a_strings: list of strings that identify Group A
    - group_b_strings: list of strings that identify Group B
    
    Returns: 'Group A', 'Group B', or the original community name
    """
    community_lower = community_name.lower()
    
    for pattern in group_a_strings:
        if pattern.lower() in community_lower:
            return 'Black, African-American'
    
    for pattern in group_b_strings:
        if pattern.lower() in community_lower:
            return 'Latine'
    
    return community_name  # Return original name if no match

# --- CONFIGURE YOUR GROUPS HERE ---
group_a_strings = ["black", "african"]  # Example: communities with these strings go to Group A
group_b_strings = ['latin']  # Example: communities with these strings go to Group B

# --- Aggregate scores by group/community ---
aggregated_scores = {model_name: {} for model_name in models}
for model_name, model_scores in community_scores.items():
    for community, scores in model_scores.items():

        if len(scores) < 10:
            continue

        group_or_community = group_community(community, group_a_strings, group_b_strings)
        
        if group_or_community not in aggregated_scores[model_name]:
            aggregated_scores[model_name][group_or_community] = []
        
        aggregated_scores[model_name][group_or_community].extend(scores)

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=["Subcommunity Analysis", "Term Analysis", "", ""],
    column_widths=[0.3, 0.3],
    horizontal_spacing=0.04,
    vertical_spacing=0.3,  # Increased spacing between rows
    specs=[[{}, {}], [{}, {}]]
)

# --- Define consistent color mapping for communities ---
all_communities = set()
for model_name in selected_models:
    all_communities.update(aggregated_scores[model_name].keys())

# Sort communities for consistent ordering
sorted_communities = sorted(all_communities)
community_color_map = {
    comm: PAUL_TOL["rainbow14"][i % len(PAUL_TOL["rainbow14"])]
    for i, comm in enumerate(sorted_communities)
}

BLUE = "#77AADD"
ORANGE = "#EEDD88"

# --- Plot each model ---
for model_idx, model_name in enumerate(selected_models):
    row = model_idx + 1
    
    # === LEFT COLUMN: Community Analysis ===
    model_agg = aggregated_scores[model_name]
    
    # Sort by mean score
    sorted_items = sorted(
        model_agg.keys(),
        key=lambda c: np.mean(model_agg[c])
    )
    
    for item in sorted_items:
        scores = np.array(model_agg[item])
        mean = np.mean(scores)
        std = np.std(scores)

        fig.add_trace(
            go.Bar(
                x=[item],
                y=[mean],
                name=item,
                marker_color=community_color_map[item],
                error_y=dict(
                    type='data',
                    array=[std],
                    visible=True,
                    color='grey',
                    thickness=1,
                    width=3
                ),
                legendgroup=f"community_{item}",
                showlegend=False
            ),
            row=row, col=1,
        )

        fig.update_xaxes(
            tickfont=dict(size=14),
            tickangle=45,
        )
    
    # === RIGHT COLUMN: Term Analysis ===
    model_scores = term_scores[model_name]
    
    # Sort by mean
    sorted_groups = sorted(model_scores.keys(), key=lambda g: np.mean(model_scores[g]))
    
    means = [np.mean(model_scores[g]) for g in sorted_groups]
    stds = [np.std(model_scores[g]) for g in sorted_groups]
    colors = [BLUE if g in qrl_terms else ORANGE for g in sorted_groups]
    
    fig.add_trace(
        go.Bar(
            x=sorted_groups,
            y=means,
            error_y=dict(type="data", array=stds, visible=True, color='grey', thickness=1, width=3),
            marker_color=colors,
            showlegend=False,
        ),
        row=row, col=2,
    )
    
    # === Identify top 3 orange (Terms B) terms ===
    orange_terms = [(g, np.mean(model_scores[g])) for g in sorted_groups if g not in qrl_terms]
    # Sort by mean score and get top 3
    top_3_orange = sorted(orange_terms, key=lambda x: x[1], reverse=True)[:3]
    top_3_orange_names = {term for term, _ in top_3_orange}
    
# Update x-axis tick labels to bold top 3 orange terms (even ticks only)
    ticktext = [
        f"<b>{term}</b>" if (term in top_3_orange_names and i % 2 == 0) 
        else (term if i % 2 == 0 else "")
        for i, term in enumerate(sorted_groups)
    ]
    
    fig.update_xaxes(
        ticktext=ticktext,
        tickvals=sorted_groups,
        tickangle=45,
        tickfont=dict(size=14),
        row=row,
        col=2
    )

# --- Add dummy traces for term analysis legend ---
fig.add_trace(
    go.Bar(
        x=[None], y=[None],
        marker_color=BLUE,
        name="Reclaimed Slurs",
        showlegend=True,
        legendgroup="terms_a",
    )
)

fig.add_trace(
    go.Bar(
        x=[None], y=[None],
        marker_color=ORANGE,
        name="Other Terms",
        showlegend=True,
        legendgroup="terms_b",
    )
)



# --- Update layout ---
fig.update_layout(
    template="plotly_white",
    width=1300,
    height=1000,
    margin=dict(l=120),
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.67, bordercolor="lightgray", borderwidth=1, bgcolor="white")
)

# Update y-axes
fig.update_yaxes(range=[0, 1], row=1, col=1)
fig.update_yaxes(range=[0, 1], row=2, col=1)
fig.update_yaxes(range=[0, 1.1], row=1, col=2)
fig.update_yaxes(range=[0, 1.1], row=2, col=2)

# Add y-axis titles
fig.update_yaxes(title_text="Toxicity Score", row=1, col=1)
fig.update_yaxes(title_text="Toxicity Score", row=2, col=1)
fig.update_yaxes(title_text="", row=1, col=2)
fig.update_yaxes(title_text="", row=2, col=2)


# --- Add horizontal line separator between rows ---
fig.add_shape(
    type="line",
    xref="paper", yref="paper",
    x0=0, y0=0.46, x1=1, y1=0.46,
    line=dict(color="black", width=1, dash="solid")
)

fig.add_annotation(
    text="<b>Dolma</b>",
    xref="paper", yref="paper",
    x=-0.08, y=0.85,
    showarrow=False,
    font=dict(size=18),
    textangle=-90,
    xanchor="center"
)

fig.add_annotation(
    text="<b>ToxiGen-HateBert</b>",
    xref="paper", yref="paper",
    x=-0.08, y=0.15,
    showarrow=False,
    font=dict(size=18),
    textangle=-90,
    xanchor="center"
)





fig.show()
#fig.write_image('tox_combined.pdf', width=1300, height=1000)